# RideBase 07 — V1 Advanced Regression

Bu notebook önceki V1 regression benchmarkını; leakage-safe feature engineering, temporal cross-validation, kontrollü hyperparameter tuning, target transformation ve blending deneyleriyle geliştirir.

Amaç DAYS ve KM tahminini iyileştirmek ve observed-only regression yaklaşımının makul sınırını ölçmektir. **R²=0.70 zorlanmaz.** Production validation hâlâ **BLOCKED** durumundadır; sonuçlar yalnız RideBase Synthetic Dataset v1.2 offline PoC sonuçlarıdır.


In [1]:
from pathlib import Path
from collections import OrderedDict
import inspect, json, math, platform, time, warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn import __version__ as sklearn_version
from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import (ExtraTreesRegressor, GradientBoostingRegressor,
                              HistGradientBoostingRegressor, RandomForestRegressor,
                              VotingRegressor)
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, median_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import ParameterSampler, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

try:
    from xgboost import XGBRegressor
except Exception:
    XGBRegressor=None
try:
    from lightgbm import LGBMRegressor
except Exception:
    LGBMRegressor=None
try:
    from catboost import CatBoostRegressor
except Exception:
    CatBoostRegressor=None

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore",message="X does not have valid feature names, but LGBMRegressor was fitted with feature names",category=UserWarning)
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)

def find_project_root():
    here=Path.cwd().resolve()
    for candidate in [here,*here.parents]:
        if (candidate/"notebooks").is_dir() and (candidate/"models").is_dir():
            return candidate
    raise FileNotFoundError("ridebase-ml proje kökü bulunamadı")

ROOT=find_project_root()
DATASET_ROOT=ROOT.parent/"ridebase_v1_2"
SOURCE=DATASET_ROOT/"source_tables"
DERIVED=DATASET_ROOT/"derived_outputs"
MODELS=ROOT/"models"; OUTPUTS=ROOT/"outputs"; REPORTS=ROOT/"reports"
TABLES=REPORTS/"tables"; FIGURES=REPORTS/"figures"/"v1_advanced_regression"
for d in [MODELS,OUTPUTS,TABLES,FIGURES]: d.mkdir(parents=True,exist_ok=True)
SEED=42; DATASET_VERSION="1.2.0"; MIN_SEGMENT_N=50

def regression_metrics(y,pred,target):
    y=np.asarray(y,float); pred=np.asarray(pred,float); ae=np.abs(pred-y)
    out={"mae":mean_absolute_error(y,pred),"median_ae":median_absolute_error(y,pred),
         "rmse":mean_squared_error(y,pred)**0.5,"r2":r2_score(y,pred),
         "bias":float(np.mean(pred-y)),"p90_ae":float(np.quantile(ae,.90))}
    tolerances=[15,30,45,60,90] if target=="DAYS" else [500,1000,1500,2000,5000]
    out.update({f"within_{x}":float(np.mean(ae<=x)) for x in tolerances})
    return out


## 1. Dataset, artifact ve split koruması

### Ne yapıyoruz?
Authoritative v1.2 metadata, 05 preprocessorı ve 06 Basic V1 model/metriklerini yüklüyoruz. TRAIN/VALIDATION/TEST splitleri manifestten gelir; random split yapılmaz.

### Neden yapıyoruz?
Advanced çalışmanın iyileşmesini aynı veri, aynı observed kapsamı ve aynı Basic V1 referansı üzerinde ölçmek gerekir.

### Leakage riski var mı?
Bu aşamada target tabloları yalnız maske ve `y` için kullanılır; X kaynağı snapshot-time featurelardır.

### Sonuç nasıl yorumlanmalı?
Guard'lardan biri geçmezse notebook eski veya uyumsuz artifactla sonuç üretmeden durur.


In [2]:
with open(DERIVED/"dataset_metadata.json",encoding="utf-8") as f: metadata=json.load(f)
info=metadata["dataset"]
if info.get("dataset_version")!=DATASET_VERSION or info.get("generator_version")!=DATASET_VERSION:
    raise RuntimeError("Yalnız RideBase Synthetic Dataset v1.2.0 kullanılabilir")

snapshots=pd.read_parquet(DERIVED/"ml_maintenance_snapshots.parquet")
targets=pd.read_parquet(DERIVED/"ml_next_service_targets.parquet")
manifest=pd.read_parquet(OUTPUTS/"ml_modeling_manifest.parquet")
split_manifest=pd.read_csv(DERIVED/"split_manifest.csv",encoding="utf-8-sig",low_memory=False)
base_preprocessor=joblib.load(MODELS/"ml_preprocessor_v1_2.joblib")
basic_days_model=joblib.load(MODELS/"v1_next_service_days_model.joblib")
basic_km_model=joblib.load(MODELS/"v1_next_service_km_model.joblib")
basic_metrics=pd.read_csv(TABLES/"v1_regression_metrics.csv")
v0_v1_basic=pd.read_csv(TABLES/"v0_vs_v1_regression_comparison.csv")
v0_predictions=pd.read_parquet(OUTPUTS/"v0_rule_baseline_next_service_predictions.parquet")

expected_splits={"TRAIN":27428,"VALIDATION":6399,"TEST":7691}
if len(snapshots)!=41518 or manifest.split.value_counts().to_dict()!=expected_splits:
    raise RuntimeError("Dataset veya split guard failed")
if len(base_preprocessor.feature_names_in_)!=127 or len(base_preprocessor.get_feature_names_out())!=277:
    raise RuntimeError("05 preprocessing contract mismatch")
for frame,name in [(snapshots,"snapshots"),(targets,"targets"),(manifest,"manifest")]:
    if frame.snapshot_id.duplicated().any(): raise RuntimeError(f"Duplicate snapshot_id: {name}")

target_cols=targets[["snapshot_id","target_event_observed","days_to_next_service","km_to_next_service","target_km_valid"]]
joined=(manifest.merge(target_cols,on="snapshot_id",how="left",validate="one_to_one",suffixes=("_manifest","_target"))
                .merge(snapshots,on=["snapshot_id","motorcycle_id"],how="left",validate="one_to_one"))
if {"snapshot_at_x","snapshot_at_y"}.issubset(joined.columns):
    left_ts=pd.to_datetime(joined.snapshot_at_x); right_ts=pd.to_datetime(joined.snapshot_at_y)
    if not left_ts.equals(right_ts): raise RuntimeError("Manifest/snapshot timestamp alignment failed")
    joined["snapshot_at"]=right_ts
    joined=joined.drop(columns=["snapshot_at_x","snapshot_at_y"])
joined=joined.set_index("snapshot_id",drop=False)
joined.index.name="_snapshot_index"
days_col="days_to_next_service_target"; km_col="km_to_next_service_target"
days_mask=(joined.v1_regression_eligible.astype(bool)&joined[days_col].notna()&np.isfinite(joined[days_col])&(joined[days_col]>=0))
km_mask=(days_mask&joined[km_col].notna()&np.isfinite(joined[km_col])&(joined[km_col]>=0)&(joined.target_km_valid_target==1))
days_counts=joined.loc[days_mask].groupby("split").size().to_dict()
km_counts=joined.loc[km_mask].groupby("split").size().to_dict()
if days_counts!={"TRAIN":20679,"VALIDATION":1932,"TEST":2622}:
    raise RuntimeError(f"Observed days contract mismatch: {days_counts}")
print("DATASET_ARTIFACT_GUARD=PASS",info["dataset_version"],expected_splits)
print("OBSERVED_DAYS=",days_counts,"VALID_KM=",km_counts)


DATASET_ARTIFACT_GUARD=PASS 1.2.0 {'TRAIN': 27428, 'VALIDATION': 6399, 'TEST': 7691}
OBSERVED_DAYS= {'TEST': 2622, 'TRAIN': 20679, 'VALIDATION': 1932} VALID_KM= {'TEST': 2622, 'TRAIN': 20678, 'VALIDATION': 1931}


## 2. Feature engineering sözleşmesi

### Ne yapıyoruz?
*Feature engineering*, snapshot anında bilinen ham/geçmiş verileri modele daha anlaşılır sinyallere dönüştürmektir. *Lag feature* önceki olayın değerini; *rolling feature* yalnız geçmişteki son birkaç olayın özetini; *interaction feature* iki mevcut sinyalin sınırlı birleşimini ifade eder.

Birçok istenen alan zaten v1.2 snapshotında bulunuyor: son servis aralıkları, rolling3 aralıklar, task/fault/repair/breakdown sayıları, component bazlı son task süreleri ve overdue alanları. Bunlar tekrar üretilmez.

### Neden yapıyoruz?
Eksik kalan 2/5 servis pencereleri, geçmiş tamamlanmış ay kullanım trendleri, geçmiş failure oranları, policy-derived due sinyalleri ve az sayıda interaction hedefteki doğrusal olmayan yapıyı yakalayabilir.

### Leakage riski var mı?
Her engineered feature için kaynak ve pencere kaydedilir. Mileage yalnız `period_end_date <= snapshot_at`; servis rollingleri yalnız aynı veya daha eski servis; policy sinyali actual next service değil V0'ın snapshot-time bakım politikası sonucudur.

### Sonuç nasıl yorumlanmalı?
Audit tablosunda `uses_future_data=False`, `available_at_snapshot=True`, `PASS` olmayan hiçbir alan modele giremez.


In [3]:
work=joined.sort_values(["motorcycle_id","snapshot_at","snapshot_id"]).copy()
eng=pd.DataFrame(index=joined.index)
audit=[]; groups=OrderedDict()

def register(name,values,group,source_tables,window,notes=""):
    if name in joined.columns or name in eng.columns:
        raise RuntimeError(f"Feature tekrar üretildi: {name}")
    eng[name]=pd.Series(values,index=values.index if hasattr(values,"index") else joined.index).reindex(joined.index)
    groups.setdefault(group,[]).append(name)
    audit.append({"feature_name":name,"feature_group":group,"source_tables":source_tables,
                  "calculation_window":window,"uses_future_data":False,"available_at_snapshot":True,
                  "leakage_status":"PASS","notes":notes})

# Service-history: current snapshot post-service state olduğu için yeni tamamlanan interval, bir sonraki servise göre geçmiştir.
wd=pd.to_numeric(work.days_since_previous_service,errors="coerce").where(lambda x:x>=0)
wk=pd.to_numeric(work.km_since_previous_service,errors="coerce").where(lambda x:x>=0)
gm=work.motorcycle_id
register("adv_previous_service_count",(work.service_sequence-1).clip(lower=0),"SERVICE_HISTORY","ml_maintenance_snapshots","service rows <= snapshot_at")
for window in [2,5]:
    register(f"adv_previous_{window}_interval_days_mean",wd.groupby(gm).transform(lambda s:s.rolling(window,min_periods=1).mean()),"SERVICE_HISTORY","ml_maintenance_snapshots",f"last {window} completed intervals <= snapshot_at")
    register(f"adv_previous_{window}_interval_km_mean",wk.groupby(gm).transform(lambda s:s.rolling(window,min_periods=1).mean()),"SERVICE_HISTORY","ml_maintenance_snapshots",f"last {window} completed intervals <= snapshot_at")
for label,func in [("median",lambda s:s.expanding(min_periods=1).median()),
                   ("std",lambda s:s.expanding(min_periods=2).std()),
                   ("min",lambda s:s.expanding(min_periods=1).min()),
                   ("max",lambda s:s.expanding(min_periods=1).max())]:
    register(f"adv_rolling_{label}_interval_days",wd.groupby(gm).transform(func),"SERVICE_HISTORY","ml_maintenance_snapshots","all completed intervals <= snapshot_at")
    register(f"adv_rolling_{label}_interval_km",wk.groupby(gm).transform(func),"SERVICE_HISTORY","ml_maintenance_snapshots","all completed intervals <= snapshot_at")
days_mean=wd.groupby(gm).transform(lambda s:s.expanding(min_periods=1).mean())
km_mean=wk.groupby(gm).transform(lambda s:s.expanding(min_periods=1).mean())
days_std=wd.groupby(gm).transform(lambda s:s.expanding(min_periods=2).std())
km_std=wk.groupby(gm).transform(lambda s:s.expanding(min_periods=2).std())
register("adv_previous_interval_days_cv",days_std/days_mean.replace(0,np.nan),"SERVICE_HISTORY","ml_maintenance_snapshots","all completed intervals <= snapshot_at","CV=std/mean")
register("adv_previous_interval_km_cv",km_std/km_mean.replace(0,np.nan),"SERVICE_HISTORY","ml_maintenance_snapshots","all completed intervals <= snapshot_at","CV=std/mean")

# Usage/mileage: only completely closed monthly rows are visible.
timeline=pd.read_csv(SOURCE/"mileage_timeline_monthly.csv",encoding="utf-8-sig",low_memory=False)
timeline["period_end_date"]=pd.to_datetime(timeline.period_end_date)
timeline=timeline.sort_values(["motorcycle_id","period_end_date"])
usage_frame=pd.DataFrame(index=work.index)
timeline_groups={k:g for k,g in timeline.groupby("motorcycle_id",sort=False)}
for motorcycle_id,sg in work.groupby("motorcycle_id",sort=False):
    tg=timeline_groups.get(motorcycle_id)
    if tg is None: continue
    ends=tg.period_end_date.to_numpy(dtype="datetime64[ns]")
    km=tg.km_added.to_numpy(float); prefix=np.r_[0.0,np.cumsum(km)]
    dates=pd.to_datetime(sg.snapshot_at).to_numpy(dtype="datetime64[ns]")
    right=np.searchsorted(ends,dates,side="right")
    for days in [30,60,90,180]:
        left=np.searchsorted(ends,dates-np.timedelta64(days,"D"),side="right")
        usage_frame.loc[sg.index,f"adv_recent_{days}d_km"]=prefix[right]-prefix[left]
    usage_frame.loc[sg.index,"adv_months_observed"]=right
    slopes=[]; vols=[]
    for r in right:
        vals=km[max(0,r-6):r]
        slopes.append(float(np.polyfit(np.arange(len(vals)),vals,1)[0]) if len(vals)>=2 else np.nan)
        vols.append(float(np.std(vals,ddof=1)/np.mean(vals)) if len(vals)>=2 and np.mean(vals)>0 else np.nan)
    usage_frame.loc[sg.index,"adv_mileage_growth_slope_6m"]=slopes
    usage_frame.loc[sg.index,"adv_usage_volatility_6m"]=vols
for col in ["adv_recent_30d_km","adv_recent_60d_km","adv_recent_90d_km","adv_recent_180d_km","adv_months_observed","adv_mileage_growth_slope_6m","adv_usage_volatility_6m"]:
    register(col,usage_frame[col],"USAGE_TRENDS","mileage_timeline_monthly","period_end_date <= snapshot_at")
for days in [30,90,180]:
    register(f"adv_avg_km_per_day_{days}d",usage_frame[f"adv_recent_{days}d_km"]/days,"USAGE_TRENDS","mileage_timeline_monthly",f"completed monthly rows in prior {days}d")
expected_90=joined.annual_km_baseline*90/365
register("adv_recent_vs_longterm_usage_ratio",usage_frame.adv_recent_90d_km/expected_90.replace(0,np.nan),"USAGE_TRENDS","mileage_timeline_monthly;ml_maintenance_snapshots","completed prior 90d / baseline expectation")

# Maintenance/failure summaries from snapshot-time cumulative state.
depth=joined.service_sequence.clip(lower=1)
for name,num in [("adv_repair_rate",joined.repair_service_count),("adv_breakdown_rate",joined.breakdown_service_count),
                 ("adv_periodic_rate",joined.periodic_service_count),("adv_failure_rate",joined.previous_failure_count),
                 ("adv_fault_task_rate",joined.total_fault_task_count)]:
    register(name,num/depth,"MAINTENANCE_FAILURE","ml_maintenance_snapshots","cumulative history <= snapshot_at")
register("adv_declined_task_rate",joined.total_declined_task_count/joined.total_task_count.replace(0,np.nan),"MAINTENANCE_FAILURE","ml_maintenance_snapshots","cumulative task history <= snapshot_at")
register("adv_completed_task_rate",joined.total_completed_task_count/joined.total_task_count.replace(0,np.nan),"MAINTENANCE_FAILURE","ml_maintenance_snapshots","cumulative task history <= snapshot_at")

failure_event=(work.current_is_breakdown.eq(1)|work.current_fault_task_count.gt(0))
last_failure_at=pd.to_datetime(work.snapshot_at).where(failure_event).groupby(work.motorcycle_id).ffill()
last_failure_km=work.snapshot_odometer_km.where(failure_event).groupby(work.motorcycle_id).ffill()
register("adv_days_since_last_failure",(pd.to_datetime(work.snapshot_at)-last_failure_at).dt.total_seconds()/86400,"MAINTENANCE_FAILURE","ml_maintenance_snapshots","latest failure/fault service <= snapshot_at")
register("adv_km_since_last_failure",work.snapshot_odometer_km-last_failure_km,"MAINTENANCE_FAILURE","ml_maintenance_snapshots","latest failure/fault service <= snapshot_at")

# Policy-derived due signals: actual future service değil, V0'ın maintenance policy + snapshot hesabıdır.
v0=v0_predictions.set_index("snapshot_id").reindex(joined.index)
policy_features={
    "adv_policy_nearest_due_days":v0.raw_predicted_days_to_next_service.clip(lower=0),
    "adv_policy_nearest_due_km":v0.raw_predicted_km_to_next_service.clip(lower=0),
    "adv_policy_overdue_days":v0.overdue_days,
    "adv_policy_overdue_km":v0.overdue_km,
    "adv_policy_count_considered":v0.policy_count_considered,
    "adv_policy_is_overdue":v0.is_overdue.astype(float),
    "adv_policy_due_within_30d":v0.raw_predicted_days_to_next_service.between(0,30).astype(float),
    "adv_policy_due_within_60d":v0.raw_predicted_days_to_next_service.between(0,60).astype(float),
}
for name,values in policy_features.items():
    register(name,values,"MAINTENANCE_FAILURE","maintenance_policies;v0_rule_baseline_next_service_predictions","policy calculation from state available at snapshot","Policy due != actual future service")

# Sınırlı interactionlar; tüm polynomial kombinasyonları üretilmez.
register("adv_age_x_annual_km",joined.motorcycle_age_years*joined.annual_km_baseline,"INTERACTIONS","ml_maintenance_snapshots","snapshot-time interaction")
register("adv_age_x_odometer",joined.motorcycle_age_years*joined.snapshot_odometer_km,"INTERACTIONS","ml_maintenance_snapshots","snapshot-time interaction")
register("adv_usage_x_load",joined.avg_ride_days_per_week*joined.load_severity_factor,"INTERACTIONS","ml_maintenance_snapshots","snapshot-time interaction")
register("adv_history_depth_x_age",eng.adv_previous_service_count*joined.motorcycle_age_years,"INTERACTIONS","ml_maintenance_snapshots","snapshot-time interaction")
register("adv_km_last_over_daily_usage",joined.km_since_previous_service/joined.avg_km_per_day_since_previous_service.replace(0,np.nan),"INTERACTIONS","ml_maintenance_snapshots","completed last interval / historical daily rate")
register("adv_days_last_over_rolling_median",joined.days_since_previous_service/eng.adv_rolling_median_interval_days.replace(0,np.nan),"INTERACTIONS","ml_maintenance_snapshots","completed last interval / past median")
register("adv_failure_rate_x_load",eng.adv_failure_rate*joined.load_severity_factor,"INTERACTIONS","ml_maintenance_snapshots","snapshot-time interaction")
register("adv_policy_due_x_recent_usage",eng.adv_policy_nearest_due_km/(usage_frame.adv_recent_90d_km+1),"INTERACTIONS","maintenance_policies;mileage_timeline_monthly","policy due / completed prior usage")

feature_audit=pd.DataFrame(audit)
feature_audit.to_csv(TABLES/"v1_advanced_feature_leakage_audit.csv",index=False,encoding="utf-8-sig")
if not feature_audit.leakage_status.eq("PASS").all() or feature_audit.uses_future_data.any():
    raise RuntimeError("Engineered feature leakage audit failed")
print("ENGINEERED_FEATURES=",eng.shape[1],"GROUPS=",{k:len(v) for k,v in groups.items()})
display(feature_audit)


ENGINEERED_FEATURES= 51 GROUPS= {'SERVICE_HISTORY': 15, 'USAGE_TRENDS': 11, 'MAINTENANCE_FAILURE': 17, 'INTERACTIONS': 8}


,feature_name,feature_group,source_tables,calculation_window,uses_future_data,available_at_snapshot,leakage_status,notes
0,adv_previous_service_count,SERVICE_HISTORY,ml_maintenance_snapshots,service rows <= snapshot_at,False,True,PASS,
1,adv_previous_2_interval_days_mean,SERVICE_HISTORY,ml_maintenance_snapshots,last 2 completed intervals <= snapshot_at,False,True,PASS,
2,adv_previous_2_interval_km_mean,SERVICE_HISTORY,ml_maintenance_snapshots,last 2 completed intervals <= snapshot_at,False,True,PASS,
3,adv_previous_5_interval_days_mean,SERVICE_HISTORY,ml_maintenance_snapshots,last 5 completed intervals <= snapshot_at,False,True,PASS,
4,adv_previous_5_interval_km_mean,SERVICE_HISTORY,ml_maintenance_snapshots,last 5 completed intervals <= snapshot_at,False,True,PASS,
5,adv_rolling_median_interval_days,SERVICE_HISTORY,ml_maintenance_snapshots,all completed intervals <= snapshot_at,False,True,PASS,
6,adv_rolling_median_interval_km,SERVICE_HISTORY,ml_maintenance_snapshots,all completed intervals <= snapshot_at,False,True,PASS,
7,adv_rolling_std_interval_days,SERVICE_HISTORY,ml_maintenance_snapshots,all completed intervals <= snapshot_at,False,True,PASS,
8,adv_rolling_std_interval_km,SERVICE_HISTORY,ml_maintenance_snapshots,all completed intervals <= snapshot_at,False,True,PASS,
9,adv_rolling_min_interval_days,SERVICE_HISTORY,ml_maintenance_snapshots,all completed intervals <= snapshot_at,False,True,PASS,


## 3. Advanced preprocessing ve feature set ablation

### Ne yapıyoruz?
05'in 277 encoded özelliğini aynen kullanıyor, yalnız yeni sayısal alanlar için TRAIN-only median imputation + scaling uyguluyoruz. 05 artifactı overwrite edilmez; advanced transformer ayrı kaydedilir.

*Feature ablation*, grupları sırayla ekleyip aynı model ailesiyle Validation değişimini ölçmektir:

- A: 05 mevcut featurelar
- B: A + service history
- C: B + usage/mileage trends
- D: C + maintenance/failure/policy
- E: D + limited interactions

### Neden yapıyoruz?
Yeni feature sayısını artırmak tek başına başarı değildir; hangi grubun gerçekten katkı verdiğini görürüz.

### Leakage riski var mı?
Yeni imputer/scaler yalnız TRAIN split featurelarından öğrenir. Target, censor sonucu ve TEST istatistiği preprocessing fit'ine girmez.

### Sonuç nasıl yorumlanmalı?
Her hedef için en düşük Validation MAE'li set sonraki tuning aşamasına geçer; DAYS ve KM aynı seti seçmek zorunda değildir.


In [4]:
base_raw_features=list(base_preprocessor.feature_names_in_)
base_encoded_names=np.asarray(base_preprocessor.get_feature_names_out(),dtype=str)
X_base_sparse=base_preprocessor.transform(joined[base_raw_features])
if not sparse.issparse(X_base_sparse) or not np.isfinite(X_base_sparse.data).all():
    raise RuntimeError("05 base transform QA failed")
X_base=X_base_sparse.toarray().astype("float32",copy=False)

train_all=joined.split.eq("TRAIN").to_numpy()
advanced_numeric=Pipeline([
    ("imputer",SimpleImputer(strategy="median",add_indicator=True,keep_empty_features=True)),
    ("scaler",StandardScaler()),
])
advanced_numeric.fit(eng.loc[train_all])
X_eng=np.asarray(advanced_numeric.transform(eng),dtype="float32")
eng_encoded_names=np.asarray(advanced_numeric.get_feature_names_out(eng.columns),dtype=str)
if not np.isfinite(X_eng).all(): raise RuntimeError("Advanced transform NaN/inf QA failed")

# İmputer indicator sütunlarını kaynak grubuna bağla.
eng_index={name:i for i,name in enumerate(eng_encoded_names)}
group_encoded={}
for group,cols in groups.items():
    idx=[]
    for i,name in enumerate(eng_encoded_names):
        if name in cols or (name.startswith("missingindicator_") and name.replace("missingindicator_","") in cols): idx.append(i)
    group_encoded[group]=idx

feature_sets=OrderedDict()
feature_sets["SET_A_BASE_05"]=[]
feature_sets["SET_B_SERVICE_HISTORY"]=["SERVICE_HISTORY"]
feature_sets["SET_C_USAGE_TRENDS"]=["SERVICE_HISTORY","USAGE_TRENDS"]
feature_sets["SET_D_MAINTENANCE_FAILURE"]=["SERVICE_HISTORY","USAGE_TRENDS","MAINTENANCE_FAILURE"]
feature_sets["SET_E_INTERACTIONS"]=["SERVICE_HISTORY","USAGE_TRENDS","MAINTENANCE_FAILURE","INTERACTIONS"]

def feature_set_matrix(name,positions=None):
    idx=[]
    for group in feature_sets[name]: idx.extend(group_encoded[group])
    matrix=X_base if not idx else np.hstack([X_base,X_eng[:,idx]]).astype("float32",copy=False)
    return matrix if positions is None else matrix[positions]

def feature_set_names(name):
    idx=[]
    for group in feature_sets[name]: idx.extend(group_encoded[group])
    return np.r_[base_encoded_names,eng_encoded_names[idx]]

advanced_preprocessor_artifact={
    "artifact_version":"V1_ADVANCED_PREPROCESSOR_1.0.0","dataset_version":DATASET_VERSION,
    "base_preprocessor_path":"models/ml_preprocessor_v1_2.joblib","base_preprocessor":base_preprocessor,
    "engineered_transformer":advanced_numeric,"engineered_raw_columns":eng.columns.tolist(),
    "engineered_encoded_names":eng_encoded_names.tolist(),"feature_sets":dict(feature_sets),
    "group_encoded_indices":group_encoded,"fit_scope":"ALL TRAIN FEATURES; NO TARGET",
}
joblib.dump(advanced_preprocessor_artifact,MODELS/"v1_advanced_preprocessor.joblib")

positions={}
for target,mask in [("DAYS",days_mask),("KM",km_mask)]:
    positions[target]={s:np.flatnonzero((joined.split.eq(s)&mask).to_numpy()) for s in expected_splits}
ysets={
    "DAYS":{s:joined.iloc[positions["DAYS"][s]][days_col].to_numpy(float) for s in expected_splits},
    "KM":{s:joined.iloc[positions["KM"][s]][km_col].to_numpy(float) for s in expected_splits},
}

ablation_rows=[]
for set_name in feature_sets:
    X=feature_set_matrix(set_name)
    row={"feature_set":set_name,"feature_count":X.shape[1],"notes":"Same default HistGradientBoosting family; full Validation"}
    for target in ["DAYS","KM"]:
        p=positions[target]
        model=HistGradientBoostingRegressor(random_state=SEED)
        model.fit(X[p["TRAIN"]],ysets[target]["TRAIN"])
        pred=np.clip(model.predict(X[p["VALIDATION"]]),0,None)
        met=regression_metrics(ysets[target]["VALIDATION"],pred,target)
        row[f"{target.lower()}_val_mae"]=met["mae"]
        row[f"{target.lower()}_val_r2"]=met["r2"]
    ablation_rows.append(row)
ablation=pd.DataFrame(ablation_rows)
ablation["incremental_gain"]=(ablation.days_val_mae.shift(1)-ablation.days_val_mae).fillna(0)
ablation.to_csv(TABLES/"v1_advanced_feature_ablation.csv",index=False,encoding="utf-8-sig")
best_feature_set={"DAYS":str(ablation.loc[ablation.days_val_mae.idxmin(),"feature_set"]),
                  "KM":str(ablation.loc[ablation.km_val_mae.idxmin(),"feature_set"])}
print("BEST_FEATURE_SETS=",best_feature_set)
display(ablation)


BEST_FEATURE_SETS= {'DAYS': 'SET_B_SERVICE_HISTORY', 'KM': 'SET_D_MAINTENANCE_FAILURE'}


,feature_set,feature_count,notes,days_val_mae,days_val_r2,km_val_mae,km_val_r2,incremental_gain
0,SET_A_BASE_05,277,Same default HistGradientBoosting family; full...,22.471394,0.283081,1482.750395,0.152122,0.000000
1,SET_B_SERVICE_HISTORY,306,Same default HistGradientBoosting family; full...,22.401203,0.275114,1536.067321,0.142180,0.070191
2,SET_C_USAGE_TRENDS,319,Same default HistGradientBoosting family; full...,23.401403,0.252984,1501.837969,0.145342,-1.000200
3,SET_D_MAINTENANCE_FAILURE,338,Same default HistGradientBoosting family; full...,23.551450,0.267477,1482.646588,0.158636,-0.150047
4,SET_E_INTERACTIONS,348,Same default HistGradientBoosting family; full...,23.039233,0.271290,1497.137754,0.153457,0.512217


## 4. Target dağılımı, dönüşüm ve outlier sözleşmesi

### Ne yapıyoruz?
TRAIN targetlarının skewness, P95, P99 ve üst kuyruklarını ölçüyoruz. *Target transformation*, sağa çarpık hedefte `log1p(y)` öğrenip tahmini `expm1` ile orijinal birime döndürmektir.

### Neden yapıyoruz?
Log dönüşümü büyük uçların optimizasyona hakim olmasını azaltabilir; ancak MAE/R² etkisi Validation üzerinde ölçülmeden seçilmez.

### Leakage riski var mı?
Dönüşüm sabit matematiksel fonksiyondur ve yalnız TRAIN y fit'inde uygulanır. P95/P99 satırları silinmez; ana metrik her zaman full Validation/TEST üzerindedir.

### Sonuç nasıl yorumlanmalı?
Central %99 metriği yalnız diagnostic'tir. R² tek başına yeterli değildir: uç değerlere duyarlıdır; bu nedenle MAE, Median AE, bias ve tolerance oranlarıyla birlikte okunur.


In [5]:
distribution_rows=[]
for target in ["DAYS","KM"]:
    y=ysets[target]["TRAIN"]
    distribution_rows.append({"target":target,"n":len(y),"mean":np.mean(y),"median":np.median(y),
                              "std":np.std(y),"skewness":pd.Series(y).skew(),"p95":np.quantile(y,.95),
                              "p99":np.quantile(y,.99),"max":np.max(y)})
target_distribution=pd.DataFrame(distribution_rows)
target_distribution.to_csv(TABLES/"v1_advanced_target_distribution.csv",index=False,encoding="utf-8-sig")
display(target_distribution)


,target,n,mean,median,std,skewness,p95,p99,max
0,DAYS,20679,159.568562,111.952778,151.178056,2.231611,450.920764,744.911986,1343.889583
1,KM,20678,6762.512139,5267.500000,5032.117089,2.336019,16560.900000,25613.230000,63341.000000


## 5. Temporal cross-validation ve kontrollü hyperparameter tuning

### Ne yapıyoruz?
*Hyperparameter tuning*, modelin ağaç derinliği veya learning rate gibi eğitimden önce belirlenen ayarlarını karşılaştırmaktır. *Cross-validation*, TRAIN'i birden fazla öğrenme/ölçme penceresine bölerek tek döneme aşırı uyumu azaltır.

TRAIN snapshotları zamana göre sıralanır ve iki expanding fold kullanılır: geçmişten öğren, daha sonraki TRAIN parçasını doğrula. Random KFold yoktur. Her ailede yalnız iki kontrollü parametre örneği denenir; devasa grid yoktur.

### Neden yapıyoruz?
HistGradientBoosting ciddi biçimde tune edilir; ExtraTrees, RandomForest, GradientBoosting, XGBoost, LightGBM ve CatBoost aileleri aynı temporal disiplinle kıyaslanır.

### Leakage riski var mı?
CV yalnız TRAIN içindedir. Harici VALIDATION; feature seti, model ailesi, raw/log ve blend seçiminde kullanılır. TEST bu bölümde hiçbir fonksiyona verilmez.

### Sonuç nasıl yorumlanmalı?
Primary CV ve Validation metriği MAE'dir. Bir aile unavailable/fail olursa status tablosuna yazılır, notebook devam eder.


In [ ]:
MODEL_SPECS=OrderedDict({
    "HistGradientBoostingRegressor":{
        "class":HistGradientBoostingRegressor,
        "base":{"random_state":SEED},
        "space":{"learning_rate":[.03,.05,.08,.1],"max_iter":[200,350,500],"max_leaf_nodes":[15,31,63],
                 "max_depth":[None,6,10],"min_samples_leaf":[10,20,40,80],"l2_regularization":[0,.1,1,5],"max_bins":[127,255]}},
    "ExtraTreesRegressor":{
        "class":ExtraTreesRegressor,"base":{"random_state":SEED,"n_jobs":-1},
        "space":{"n_estimators":[120,200],"max_depth":[None,18,30],"min_samples_leaf":[1,2,5],"max_features":[.6,.8,1.0,"sqrt"],"bootstrap":[False,True]}},
    "RandomForestRegressor":{
        "class":RandomForestRegressor,"base":{"random_state":SEED,"n_jobs":-1},
        "space":{"n_estimators":[120,200],"max_depth":[None,18,30],"min_samples_leaf":[1,2,5],"max_features":[.6,.8,"sqrt"],"bootstrap":[True]}},
    "GradientBoostingRegressor":{
        "class":GradientBoostingRegressor,"base":{"random_state":SEED},
        "space":{"n_estimators":[100,160],"learning_rate":[.03,.05,.08],"max_depth":[2,3],"min_samples_leaf":[5,15,30],"max_features":[.8,1.0]}},
    "XGBRegressor":{
        "class":XGBRegressor,"base":{"random_state":SEED,"n_jobs":-1,"tree_method":"hist","verbosity":0,"objective":"reg:squarederror"},
        "space":{"n_estimators":[250,400],"learning_rate":[.03,.05,.08],"max_depth":[3,5,7],"min_child_weight":[1,5,10],
                 "subsample":[.7,.9,1.0],"colsample_bytree":[.7,.9,1.0],"reg_alpha":[0,.1,1],"reg_lambda":[1,5,10]}},
    "LGBMRegressor":{
        "class":LGBMRegressor,"base":{"random_state":SEED,"n_jobs":-1,"verbosity":-1},
        "space":{"n_estimators":[250,400],"learning_rate":[.03,.05,.08],"num_leaves":[15,31,63],"max_depth":[-1,6,10],
                 "min_child_samples":[10,20,40],"subsample":[.8,1.0],"colsample_bytree":[.7,.9,1.0],"reg_alpha":[0,.1,1],"reg_lambda":[0,1,5]}},
    "CatBoostRegressor":{
        "class":CatBoostRegressor,"base":{"random_seed":SEED,"verbose":0,"thread_count":-1,"allow_writing_files":False,"loss_function":"RMSE"},
        "space":{"iterations":[250,400],"learning_rate":[.03,.05,.08],"depth":[5,7,9],"l2_leaf_reg":[1,3,7],"random_strength":[0,.5,1]}},
})

def make_model(family,params=None):
    spec=MODEL_SPECS[family]
    if spec["class"] is None: raise ImportError(f"{family} unavailable")
    kwargs={**spec["base"],**(params or {})}
    return spec["class"](**kwargs)

def wrap_target(estimator,transform):
    if transform=="LOG1P":
        return TransformedTargetRegressor(regressor=estimator,func=np.log1p,inverse_func=np.expm1,check_inverse=False)
    return estimator

def temporal_training_view(target,set_name):
    pos=positions[target]["TRAIN"]
    order=np.argsort(pd.to_datetime(joined.iloc[pos].snapshot_at).to_numpy())
    X=feature_set_matrix(set_name,pos)[order]
    y=ysets[target]["TRAIN"][order]
    n=len(y); a=n//2; b=3*n//4
    cv=[(np.arange(0,a),np.arange(a,b)),(np.arange(0,b),np.arange(b,n))]
    return X,y,cv

candidate_store={"DAYS":{},"KM":{}}
tuning_rows={"DAYS":[],"KM":[]}
status_rows=[]

# Basic V1 harici Validation referansı da aday havuzunda tutulur; advanced deneyler kötüleşirse sonuç saklanmaz.
for target,basic_model in [("DAYS",basic_days_model),("KM",basic_km_model)]:
    p=positions[target]
    pred=np.clip(basic_model.predict(X_base[p["VALIDATION"]]),0,None)
    met=regression_metrics(ysets[target]["VALIDATION"],pred,target)
    candidate_store[target]["BasicV1Reference__RAW"]={"family":"BasicV1Reference","transform":"RAW","model":basic_model,
        "pred_val":pred,"metrics":met,"params":{},"feature_set":"SET_A_BASE_05"}
    tuning_rows[target].append({"model":"BasicV1Reference","target_transform":"RAW","params":"{}",**{f"validation_{k}":v for k,v in met.items()},"runtime":0.0,"status":"REFERENCE"})

for target in ["DAYS","KM"]:
    set_name=best_feature_set[target]
    Xtrain,ytrain,cv=temporal_training_view(target,set_name)
    Xval=feature_set_matrix(set_name,positions[target]["VALIDATION"])
    yval=ysets[target]["VALIDATION"]
    for family,spec in MODEL_SPECS.items():
        start=time.time()
        if spec["class"] is None:
            status_rows.append({"target":target,"model":family,"available":False,"trained":False,"reason":"Package unavailable"})
            tuning_rows[target].append({"model":family,"target_transform":"N/A","params":"{}","validation_mae":np.nan,"validation_median_ae":np.nan,"validation_r2":np.nan,"runtime":0.0,"status":"UNAVAILABLE"})
            continue
        try:
            search=RandomizedSearchCV(make_model(family),spec["space"],n_iter=2,scoring="neg_mean_absolute_error",
                                      cv=cv,random_state=SEED,n_jobs=1,refit=True,error_score=np.nan,return_train_score=False)
            search.fit(Xtrain,ytrain)
            best_params=search.best_params_
            for transform in ["RAW","LOG1P"]:
                model=wrap_target(make_model(family,best_params),transform)
                fit_start=time.time(); model.fit(Xtrain,ytrain)
                pred_raw=np.asarray(model.predict(Xval),float); pred=np.clip(pred_raw,0,None)
                met=regression_metrics(yval,pred,target)
                name=f"{family}__{transform}"
                candidate_store[target][name]={"family":family,"transform":transform,"model":model,"pred_val":pred,
                    "pred_val_raw":pred_raw,"metrics":met,"params":best_params,"feature_set":set_name}
                tuning_rows[target].append({"model":family,"target_transform":transform,"params":json.dumps(best_params,sort_keys=True,default=str),
                    **{f"validation_{k}":v for k,v in met.items()},"runtime":time.time()-fit_start,
                    "cv_best_mae":-float(search.best_score_),"status":"SUCCESS"})
            status_rows.append({"target":target,"model":family,"available":True,"trained":True,"reason":"Temporal RandomizedSearchCV complete"})
        except Exception as exc:
            status_rows.append({"target":target,"model":family,"available":True,"trained":False,"reason":f"{type(exc).__name__}: {str(exc)[:180]}"})
            tuning_rows[target].append({"model":family,"target_transform":"N/A","params":"{}","validation_mae":np.nan,"validation_median_ae":np.nan,"validation_r2":np.nan,"runtime":time.time()-start,"status":"FAILED"})

days_tuning=pd.DataFrame(tuning_rows["DAYS"]).sort_values(["validation_mae","validation_median_ae"],na_position="last")
km_tuning=pd.DataFrame(tuning_rows["KM"]).sort_values(["validation_mae","validation_median_ae"],na_position="last")
days_tuning.to_csv(TABLES/"v1_advanced_model_tuning_days.csv",index=False,encoding="utf-8-sig")
km_tuning.to_csv(TABLES/"v1_advanced_model_tuning_km.csv",index=False,encoding="utf-8-sig")
model_status=pd.DataFrame(status_rows)
model_status.to_csv(TABLES/"v1_advanced_model_status.csv",index=False,encoding="utf-8-sig")
print("DAYS TUNING TOP"); display(days_tuning.head(10))
print("KM TUNING TOP"); display(km_tuning.head(10))
display(model_status)


## 6. Ensemble, blending ve segment model deneyi

### Ne yapıyoruz?
*Ensemble* birden fazla modelin tahminini birleştirir. *Blending* burada farklı ailelerin basit ortalaması veya yalnız Validation MAE'lerinden türetilmiş ters-hata ağırlıklı ortalamasıdır.

Ayrıca yeterli örnekli `ilk 1–2 servis` ve `3+ servis` gruplarında ayrı global-aile modellerinin Validation faydasını diagnostic olarak ölçüyoruz. Görülmeyen/yetersiz grupta global tahmin fallback'tir.

### Neden yapıyoruz?
Farklı model aileleri farklı hata örüntülerini dengeleyebilir. Ancak basit blending yeterliyse stacking yapılmaz; out-of-fold meta-model gerektiren gereksiz karmaşıklık eklenmez.

### Leakage riski var mı?
Üyeler TRAIN'de fit edilmiştir; aile, transform ve ağırlıklar yalnız VALIDATION ile seçilir. TEST ağırlık veya segment kararına girmez.

### Sonuç nasıl yorumlanmalı?
Ensemble yalnız Validation MAE/Median AE sıralamasında gerçekten öne çıkarsa final aday olur.


In [ ]:
ensemble_rows=[]
for target in ["DAYS","KM"]:
    # Her aileden en iyi transform; Basic referans blend üyesi değildir çünkü feature boyutu farklı olabilir.
    tuned=[(name,rec) for name,rec in candidate_store[target].items() if rec["family"]!="BasicV1Reference"]
    family_best=[]
    for family in MODEL_SPECS:
        options=[x for x in tuned if x[1]["family"]==family]
        if options: family_best.append(min(options,key=lambda x:(x[1]["metrics"]["mae"],x[1]["metrics"]["median_ae"])))
    family_best=sorted(family_best,key=lambda x:(x[1]["metrics"]["mae"],x[1]["metrics"]["median_ae"]))[:3]
    if len(family_best)>=2:
        names=[x[0] for x in family_best]
        member_transforms={x[1]["transform"] for x in family_best}
        ensemble_transform=next(iter(member_transforms)) if len(member_transforms)==1 else "MIXED"
        # VotingRegressor üyelerin raw tahminlerini ortalar, sonra ürün kontratı sıfıra clip eder.
        preds_raw=np.vstack([x[1]["pred_val_raw"] for x in family_best])
        weights_simple=np.repeat(1/len(names),len(names))
        maes=np.array([x[1]["metrics"]["mae"] for x in family_best]); inv=1/maes; weights_weighted=inv/inv.sum()
        for ens_name,weights in [("SIMPLE_AVERAGE",weights_simple),("INVERSE_VALIDATION_MAE_BLEND",weights_weighted)]:
            pred=np.clip(np.average(preds_raw,axis=0,weights=weights),0,None); met=regression_metrics(ysets[target]["VALIDATION"],pred,target)
            key=f"ENSEMBLE__{ens_name}"
            candidate_store[target][key]={"family":"ENSEMBLE","transform":ensemble_transform,"model":None,"pred_val":pred,"metrics":met,
                "params":{},"feature_set":best_feature_set[target],"members":names,"weights":weights.tolist()}
            ensemble_rows.append({"target":target,"ensemble_name":ens_name,"members":" | ".join(names),
                                  "weights":json.dumps(weights.round(6).tolist()),"validation_mae":met["mae"],
                                  "validation_r2":met["r2"],"selected":False})

candidate_rankings={}
selected_name={}
for target in ["DAYS","KM"]:
    rows=[]
    for name,rec in candidate_store[target].items():
        rows.append({"candidate":name,"family":rec["family"],"target_transform":rec["transform"],"feature_set":rec["feature_set"],**rec["metrics"]})
    rank=pd.DataFrame(rows).sort_values(["mae","median_ae","candidate"]).reset_index(drop=True)
    rank["rank"]=np.arange(1,len(rank)+1); candidate_rankings[target]=rank
    selected_name[target]=str(rank.iloc[0].candidate)
    print(target,"VALIDATION FINAL CANDIDATE RANKING"); display(rank.head(12))

for row in ensemble_rows:
    row["selected"]=selected_name[row["target"]]==f"ENSEMBLE__{row['ensemble_name']}"
ensemble_results=pd.DataFrame(ensemble_rows,columns=["target","ensemble_name","members","weights","validation_mae","validation_r2","selected"])
ensemble_results.to_csv(TABLES/"v1_advanced_ensemble_results.csv",index=False,encoding="utf-8-sig")

# Segment deneyi: en iyi tek global adayla aynı aile/transform, history depth'e göre iki fit.
segment_experiment=[]
for target in ["DAYS","KM"]:
    singles=candidate_rankings[target][candidate_rankings[target].family!="ENSEMBLE"]
    best_single_name=str(singles.iloc[0].candidate); rec=candidate_store[target][best_single_name]
    set_name=rec["feature_set"]; X=feature_set_matrix(set_name)
    p=positions[target]; train_pos=p["TRAIN"]; val_pos=p["VALIDATION"]
    train_seg=np.where(joined.iloc[train_pos].service_sequence.to_numpy()<=2,"FIRST_SECOND","DEEP_HISTORY")
    val_seg=np.where(joined.iloc[val_pos].service_sequence.to_numpy()<=2,"FIRST_SECOND","DEEP_HISTORY")
    seg_pred=rec["pred_val"].copy(); fitted_segments=0
    if rec["family"]=="BasicV1Reference": family="HistGradientBoostingRegressor"; params={}; transform="RAW"
    else: family=rec["family"]; params=rec["params"]; transform=rec["transform"]
    for segment in ["FIRST_SECOND","DEEP_HISTORY"]:
        tr=np.flatnonzero(train_seg==segment); va=np.flatnonzero(val_seg==segment)
        if len(tr)>=2000 and len(va)>=200:
            sm=wrap_target(make_model(family,params),transform); sm.fit(X[train_pos[tr]],ysets[target]["TRAIN"][tr])
            seg_pred[va]=np.clip(sm.predict(X[val_pos[va]]),0,None); fitted_segments+=1
    global_met=rec["metrics"]; seg_met=regression_metrics(ysets[target]["VALIDATION"],seg_pred,target)
    segment_experiment.append({"target":target,"base_candidate":best_single_name,"fitted_segments":fitted_segments,
                               "global_validation_mae":global_met["mae"],"segmented_validation_mae":seg_met["mae"],
                               "relative_improvement_pct":(global_met["mae"]-seg_met["mae"])/global_met["mae"]*100,
                               "selected":False,"notes":"Diagnostic only; global fallback. Final selection remains common pipeline."})
pd.DataFrame(segment_experiment).to_csv(TABLES/"v1_advanced_segment_model_experiment.csv",index=False,encoding="utf-8-sig")


DAYS VALIDATION FINAL CANDIDATE RANKING


,candidate,family,target_transform,feature_set,mae,median_ae,rmse,r2,bias,p90_ae,within_15,within_30,within_45,within_60,within_90,rank
0,ENSEMBLE__INVERSE_VALIDATION_MAE_BLEND,ENSEMBLE,MIXED,SET_B_SERVICE_HISTORY,21.696521,15.480194,29.636060,0.317002,-0.908449,49.052194,0.482919,0.793478,0.884576,0.935818,0.984990,1
1,ENSEMBLE__SIMPLE_AVERAGE,ENSEMBLE,MIXED,SET_B_SERVICE_HISTORY,21.729251,15.525310,29.636044,0.317003,-0.895971,49.231988,0.478261,0.791925,0.884058,0.935300,0.984990,2
2,GradientBoostingRegressor__RAW,GradientBoostingRegressor,RAW,SET_B_SERVICE_HISTORY,21.754273,15.557810,30.967679,0.254246,1.656431,49.349447,0.484990,0.775880,0.882505,0.934265,0.977226,3
3,BasicV1Reference__RAW,BasicV1Reference,RAW,SET_A_BASE_05,22.471394,16.723599,30.363077,0.283081,2.435359,50.106080,0.450311,0.777433,0.871118,0.933230,0.984990,4
4,LGBMRegressor__RAW,LGBMRegressor,RAW,SET_B_SERVICE_HISTORY,22.752542,14.730656,31.691771,0.218963,-7.982369,56.000482,0.509317,0.743271,0.848861,0.916667,0.983954,5
5,HistGradientBoostingRegressor__RAW,HistGradientBoostingRegressor,RAW,SET_B_SERVICE_HISTORY,23.554058,19.197317,30.174360,0.291965,3.660359,48.050989,0.370083,0.763458,0.889752,0.939959,0.988095,6
6,CatBoostRegressor__RAW,CatBoostRegressor,RAW,SET_B_SERVICE_HISTORY,24.379884,18.743657,32.772751,0.164774,6.966309,49.198545,0.377329,0.758282,0.880435,0.926501,0.977743,7
7,XGBRegressor__RAW,XGBRegressor,RAW,SET_B_SERVICE_HISTORY,24.959727,15.761526,34.886012,0.053586,-12.366657,60.653695,0.481884,0.701863,0.816770,0.897516,0.975155,8
8,CatBoostRegressor__LOG1P,CatBoostRegressor,LOG1P,SET_B_SERVICE_HISTORY,25.265674,15.023591,35.976930,-0.006530,-18.233792,63.440129,0.499482,0.671325,0.797619,0.884058,0.972567,9
9,RandomForestRegressor__LOG1P,RandomForestRegressor,LOG1P,SET_B_SERVICE_HISTORY,25.471457,20.166601,33.164539,0.144685,8.584688,50.429997,0.363872,0.714286,0.863354,0.930642,0.981884,10


KM VALIDATION FINAL CANDIDATE RANKING


,candidate,family,target_transform,feature_set,mae,median_ae,rmse,r2,bias,p90_ae,within_500,within_1000,within_1500,within_2000,within_5000,rank
0,GradientBoostingRegressor__RAW,GradientBoostingRegressor,RAW,SET_D_MAINTENANCE_FAILURE,1473.692781,903.066422,2220.696548,0.172997,-472.707441,3551.174190,0.284827,0.539099,0.678405,0.766442,0.954946,1
1,BasicV1Reference__RAW,BasicV1Reference,RAW,SET_A_BASE_05,1482.750395,944.124910,2248.549506,0.152122,-358.129291,3515.028467,0.285862,0.526670,0.686173,0.772657,0.955981,2
2,ENSEMBLE__INVERSE_VALIDATION_MAE_BLEND,ENSEMBLE,MIXED,SET_D_MAINTENANCE_FAILURE,1485.390120,888.752480,2264.310458,0.140194,-560.815523,3620.665563,0.288969,0.543760,0.689280,0.766960,0.953910,3
3,ENSEMBLE__SIMPLE_AVERAGE,ENSEMBLE,MIXED,SET_D_MAINTENANCE_FAILURE,1486.052906,889.396449,2265.496861,0.139293,-562.842235,3620.201470,0.291559,0.542724,0.688762,0.766442,0.953392,4
4,HistGradientBoostingRegressor__RAW,HistGradientBoostingRegressor,RAW,SET_D_MAINTENANCE_FAILURE,1511.415823,924.067427,2283.542411,0.125527,-502.135408,3617.137674,0.275505,0.533920,0.678923,0.763853,0.955981,5
5,LGBMRegressor__RAW,LGBMRegressor,RAW,SET_D_MAINTENANCE_FAILURE,1550.865352,945.337654,2361.767351,0.064589,-712.071741,3847.422662,0.299845,0.519420,0.667530,0.742620,0.947695,6
6,CatBoostRegressor__LOG1P,CatBoostRegressor,LOG1P,SET_D_MAINTENANCE_FAILURE,1560.943282,896.544079,2428.328324,0.011121,-1000.390122,4014.719109,0.302952,0.540135,0.669083,0.746763,0.942517,7
7,RandomForestRegressor__LOG1P,RandomForestRegressor,LOG1P,SET_D_MAINTENANCE_FAILURE,1596.552305,1208.553959,2241.835531,0.157178,-44.893654,3437.567525,0.193164,0.397721,0.619368,0.773175,0.961160,8
8,XGBRegressor__RAW,XGBRegressor,RAW,SET_D_MAINTENANCE_FAILURE,1679.738808,1126.153320,2439.323171,0.002146,-717.747932,3986.535889,0.243915,0.452097,0.611082,0.721906,0.943035,9
9,CatBoostRegressor__RAW,CatBoostRegressor,RAW,SET_D_MAINTENANCE_FAILURE,1680.703074,1332.887082,2272.596514,0.133890,206.712178,3403.038826,0.182289,0.363024,0.566028,0.723459,0.964267,10


## 7. Validation seçimi, reproducibility ve final TEST

### Ne yapıyoruz?
Feature seti, model ailesi, hyperparameter, raw/log ve blend kararı artık kilitlenir. Final estimator TRAIN'in tamamında iki kez fit edilir; VALIDATION tahminleri `1e-7` toleransta aynı olmalıdır.

### Neden yapıyoruz?
*Overfitting*, TRAIN performansı iyi görünürken daha sonraki dönemde bozulmadır. Validation→TEST farkını sonradan raporlarız; TEST'e göre model değiştirmeyiz.

### Leakage riski var mı?
TEST matrisi yalnız bu hücrede, bütün seçimler kilitlendikten sonra final değerlendirme için kullanılır. Reproducibility kontrolü TEST'e açılmadan VALIDATION üzerinde yapılır.

### Sonuç nasıl yorumlanmalı?
Final ana tahmin ürün-kullanılabilir `max(pred,0)` değeridir; negatif raw tahmin sayısı ayrıca raporlanır.


In [ ]:
def fresh_candidate_estimator(target,name):
    rec=candidate_store[target][name]
    if rec["family"]=="BasicV1Reference":
        return HistGradientBoostingRegressor(random_state=SEED)
    if rec["family"]=="ENSEMBLE":
        estimators=[]
        for i,member_name in enumerate(rec["members"]):
            member=candidate_store[target][member_name]
            estimators.append((f"m{i}_{member['family']}",wrap_target(make_model(member["family"],member["params"]),member["transform"])))
        return VotingRegressor(estimators=estimators,weights=rec["weights"],n_jobs=1)
    return wrap_target(make_model(rec["family"],rec["params"]),rec["transform"])

final_models={}; final_predictions={}; reproducibility={}; final_metrics_rows=[]
for target in ["DAYS","KM"]:
    name=selected_name[target]; rec=candidate_store[target][name]; set_name=rec["feature_set"]
    X=feature_set_matrix(set_name); p=positions[target]
    model1=fresh_candidate_estimator(target,name); model2=fresh_candidate_estimator(target,name)
    train_order=np.argsort(pd.to_datetime(joined.iloc[p["TRAIN"]].snapshot_at).to_numpy())
    Xtrain=X[p["TRAIN"]][train_order]; ytrain=ysets[target]["TRAIN"][train_order]
    model1.fit(Xtrain,ytrain); model2.fit(Xtrain,ytrain)
    val1=np.asarray(model1.predict(X[p["VALIDATION"]]),float); val2=np.asarray(model2.predict(X[p["VALIDATION"]]),float)
    reproducibility[target]=bool(np.allclose(val1,val2,rtol=1e-7,atol=1e-7))
    if not reproducibility[target]: raise RuntimeError(f"{target} reproducibility failed")
    refit_mae=regression_metrics(ysets[target]["VALIDATION"],np.clip(val1,0,None),target)["mae"]
    if not np.isclose(refit_mae,rec["metrics"]["mae"],rtol=1e-6,atol=1e-6):
        raise RuntimeError(f"{target} selected Validation MAE was not reproduced: {refit_mae} vs {rec['metrics']['mae']}")
    # TEST ilk ve tek final kullanımı.
    test_raw=np.asarray(model1.predict(X[p["TEST"]]),float); test_pred=np.clip(test_raw,0,None)
    final_models[target]=model1
    final_predictions[(target,"TEST_RAW")]=test_raw; final_predictions[(target,"TEST")]=test_pred
    final_predictions[(target,"VALIDATION")]=np.clip(val1,0,None)
    for split,pred in [("VALIDATION",final_predictions[(target,"VALIDATION")]),("TEST",test_pred)]:
        met=regression_metrics(ysets[target][split],pred,target)
        for metric,value in met.items():
            final_metrics_rows.append({"target":target,"split":split,"metric":metric,"value":value,"n":len(pred),"model":name,"feature_set":set_name})
    final_metrics_rows.append({"target":target,"split":"TEST","metric":"negative_prediction_count","value":int((test_raw<0).sum()),"n":len(test_raw),"model":name,"feature_set":set_name})

advanced_metrics=pd.DataFrame(final_metrics_rows)
advanced_metrics.to_csv(TABLES/"v1_advanced_regression_metrics.csv",index=False,encoding="utf-8-sig")
joblib.dump(final_models["DAYS"],MODELS/"v1_advanced_days_model.joblib")
joblib.dump(final_models["KM"],MODELS/"v1_advanced_km_model.joblib")

def adv_metric(target,metric,split="TEST"):
    q=advanced_metrics[(advanced_metrics.target==target)&(advanced_metrics.split==split)&(advanced_metrics.metric==metric)]
    return float(q.iloc[0].value)

test_context=joined.iloc[positions["DAYS"]["TEST"]].reset_index(drop=True)
test_predictions=pd.DataFrame({
    "snapshot_id":test_context.snapshot_id,"motorcycle_id":test_context.motorcycle_id,
    "actual_days":ysets["DAYS"]["TEST"],"predicted_days":final_predictions[("DAYS","TEST")],
    "actual_km":ysets["KM"]["TEST"],"predicted_km":final_predictions[("KM","TEST")],
    "split":"TEST","dataset_version":DATASET_VERSION,"days_model":selected_name["DAYS"],"km_model":selected_name["KM"]})
test_predictions["days_error"]=test_predictions.predicted_days-test_predictions.actual_days
test_predictions["days_abs_error"]=test_predictions.days_error.abs()
test_predictions["km_error"]=test_predictions.predicted_km-test_predictions.actual_km
test_predictions["km_abs_error"]=test_predictions.km_error.abs()
test_predictions.to_parquet(OUTPUTS/"v1_advanced_regression_test_predictions.parquet",index=False)
print("VALIDATION_SELECTION_LOCKED=",selected_name)
print("REPRODUCIBILITY=",reproducibility)
display(advanced_metrics)


VALIDATION_SELECTION_LOCKED= {'DAYS': 'ENSEMBLE__INVERSE_VALIDATION_MAE_BLEND', 'KM': 'GradientBoostingRegressor__RAW'}
REPRODUCIBILITY= {'DAYS': True, 'KM': True}


,target,split,metric,value,n,model,feature_set
0,DAYS,VALIDATION,mae,21.696521,1932,ENSEMBLE__INVERSE_VALIDATION_MAE_BLEND,SET_B_SERVICE_HISTORY
1,DAYS,VALIDATION,median_ae,15.480194,1932,ENSEMBLE__INVERSE_VALIDATION_MAE_BLEND,SET_B_SERVICE_HISTORY
2,DAYS,VALIDATION,rmse,29.636060,1932,ENSEMBLE__INVERSE_VALIDATION_MAE_BLEND,SET_B_SERVICE_HISTORY
3,DAYS,VALIDATION,r2,0.317002,1932,ENSEMBLE__INVERSE_VALIDATION_MAE_BLEND,SET_B_SERVICE_HISTORY
4,DAYS,VALIDATION,bias,-0.908449,1932,ENSEMBLE__INVERSE_VALIDATION_MAE_BLEND,SET_B_SERVICE_HISTORY
5,DAYS,VALIDATION,p90_ae,49.052194,1932,ENSEMBLE__INVERSE_VALIDATION_MAE_BLEND,SET_B_SERVICE_HISTORY
6,DAYS,VALIDATION,within_15,0.482919,1932,ENSEMBLE__INVERSE_VALIDATION_MAE_BLEND,SET_B_SERVICE_HISTORY
7,DAYS,VALIDATION,within_30,0.793478,1932,ENSEMBLE__INVERSE_VALIDATION_MAE_BLEND,SET_B_SERVICE_HISTORY
8,DAYS,VALIDATION,within_45,0.884576,1932,ENSEMBLE__INVERSE_VALIDATION_MAE_BLEND,SET_B_SERVICE_HISTORY
9,DAYS,VALIDATION,within_60,0.935818,1932,ENSEMBLE__INVERSE_VALIDATION_MAE_BLEND,SET_B_SERVICE_HISTORY


## 8. Basic V1 karşılaştırması, outlier ve population-shift diagnostics

### Ne yapıyoruz?
V0 Rule, V1 Basic ve V1 Advanced sonuçlarını aynı observed TEST kapsamıyla yan yana koyuyoruz. P95/P99 uçları silmeden full ve kuyruk hatalarını ayrı ölçüyoruz.

Ayrıca V1'e giren observed population ile regression dışında kalan censored/cutoff population'ın yaş, odometre, kullanım ve geçmiş derinliği dağılımlarını karşılaştırıyoruz.

### Neden yapıyoruz?
Advanced skor iyileşse bile observed-only örnek tüm 41.518 snapshotı temsil etmeyebilir. *Diminishing returns*, ek karmaşıklığın artık küçük kazanç üretmesidir; karar MAE, R², stability ve ablation birlikte okunarak verilir.

### Leakage riski var mı?
Bu bölüm final seçim kilitlendikten sonra TEST'i yalnız raporlar. Population shift target üretmez ve modele geri beslenmez.

### Sonuç nasıl yorumlanmalı?
Büyük standardized mean difference (SMD) veya usage dağılım farkı, V1 sonucunun tüm population'a doğrudan genellenemeyeceğini gösterir.


In [ ]:
def basic_metric(target,metric,split="TEST"):
    q=basic_metrics[(basic_metrics.target==target)&(basic_metrics.split==split)&(basic_metrics.metric==metric)&(basic_metrics.notes=="ACTIONABLE")]
    return float(q.iloc[0].value)

comparison_rows=[]
for target in ["DAYS","KM"]:
    v0_target=v0_v1_basic[v0_v1_basic.target==target].set_index("metric")
    for model_name in ["V0 Rule","V1 Basic","V1 Advanced"]:
        for metric in (["mae","median_ae","r2","within_30","within_60"] if target=="DAYS" else ["mae","median_ae","r2","within_1000","within_2000"]):
            if model_name=="V0 Rule": value=float(v0_target.loc[metric,"v0_value"])
            elif model_name=="V1 Basic": value=basic_metric(target,metric)
            else: value=adv_metric(target,metric)
            comparison_rows.append({"target":target,"model":model_name,"metric":metric,"value":value})
comparison=pd.DataFrame(comparison_rows)
comparison.to_csv(TABLES/"v0_v1_advanced_regression_comparison.csv",index=False,encoding="utf-8-sig")

outlier_rows=[]
for target in ["DAYS","KM"]:
    y=ysets[target]["TEST"]; pred=final_predictions[(target,"TEST")]
    p95=np.quantile(ysets[target]["TRAIN"],.95); p99=np.quantile(ysets[target]["TRAIN"],.99)
    masks={"FULL":np.ones(len(y),bool),"CENTRAL_99":y<=p99,"ABOVE_TRAIN_P95":y>p95,"ABOVE_TRAIN_P99":y>p99}
    for group,mask in masks.items():
        met=regression_metrics(y[mask],pred[mask],target) if mask.sum()>=2 else {"mae":np.nan,"r2":np.nan}
        outlier_rows.append({"target":target,"group":group,"n":int(mask.sum()),"mae":met["mae"],"r2":met["r2"],"train_threshold":p99 if "99" in group else p95 if "95" in group else np.nan})
outlier_analysis=pd.DataFrame(outlier_rows)
outlier_analysis.to_csv(TABLES/"v1_advanced_outlier_analysis.csv",index=False,encoding="utf-8-sig")

shift_features={
    "motorcycle_age_years":joined.motorcycle_age_years,
    "snapshot_odometer_km":joined.snapshot_odometer_km,
    "annual_km_baseline":joined.annual_km_baseline,
    "service_sequence":joined.service_sequence,
    "avg_ride_days_per_week":joined.avg_ride_days_per_week,
    "adv_recent_90d_km":eng.adv_recent_90d_km,
    "adv_previous_service_count":eng.adv_previous_service_count,
}
eligible=joined.v1_regression_eligible.astype(bool)
shift_rows=[]
for name,series in shift_features.items():
    a=pd.to_numeric(series[eligible],errors="coerce").dropna(); b=pd.to_numeric(series[~eligible],errors="coerce").dropna()
    pooled=np.sqrt((a.var(ddof=1)+b.var(ddof=1))/2); smd=(a.mean()-b.mean())/pooled if pooled>0 else 0
    shift_rows.append({"feature":name,"metric":"STANDARDIZED_MEAN_DIFFERENCE","observed_value":a.mean(),"excluded_value":b.mean(),"shift":smd,"abs_shift":abs(smd),"notes":"V1 eligible vs censored/cutoff-excluded"})
obs_dist=joined.loc[eligible,"usage_type"].value_counts(normalize=True)
exc_dist=joined.loc[~eligible,"usage_type"].value_counts(normalize=True)
cats=obs_dist.index.union(exc_dist.index); tvd=.5*sum(abs(obs_dist.get(c,0)-exc_dist.get(c,0)) for c in cats)
shift_rows.append({"feature":"usage_type","metric":"TOTAL_VARIATION_DISTANCE","observed_value":np.nan,"excluded_value":np.nan,"shift":tvd,"abs_shift":abs(tvd),"notes":"distribution distance"})
population_shift=pd.DataFrame(shift_rows).sort_values("abs_shift",ascending=False)
population_shift.to_csv(TABLES/"v1_advanced_observed_censored_shift.csv",index=False,encoding="utf-8-sig")
shift_status="MATERIAL" if population_shift.abs_shift.max()>=.20 else "LIMITED"
display(comparison); display(outlier_analysis); display(population_shift)


,target,model,metric,value
0,DAYS,V0 Rule,mae,67.410541
1,DAYS,V0 Rule,median_ae,57.847917
2,DAYS,V0 Rule,r2,-2.793433
3,DAYS,V0 Rule,within_30,0.173150
4,DAYS,V0 Rule,within_60,0.519832
5,DAYS,V1 Basic,mae,23.041618
6,DAYS,V1 Basic,median_ae,17.758210
7,DAYS,V1 Basic,r2,0.417213
8,DAYS,V1 Basic,within_30,0.763921
9,DAYS,V1 Basic,within_60,0.937834


,target,group,n,mae,r2,train_threshold
0,DAYS,FULL,2622,22.205754,0.434891,NaN
1,DAYS,CENTRAL_99,2622,22.205754,0.434891,744.911986
2,DAYS,ABOVE_TRAIN_P95,0,NaN,NaN,450.920764
3,DAYS,ABOVE_TRAIN_P99,0,NaN,NaN,744.911986
4,KM,FULL,2622,1510.175608,0.248144,NaN
5,KM,CENTRAL_99,2622,1510.175608,0.248144,25613.230000
6,KM,ABOVE_TRAIN_P95,3,11699.432133,-105.529299,16560.900000
7,KM,ABOVE_TRAIN_P99,0,NaN,NaN,25613.230000


,feature,metric,observed_value,excluded_value,shift,abs_shift,notes
2,annual_km_baseline,STANDARDIZED_MEAN_DIFFERENCE,21494.225379,13976.759718,0.747271,0.747271,V1 eligible vs censored/cutoff-excluded
5,adv_recent_90d_km,STANDARDIZED_MEAN_DIFFERENCE,5190.005548,3578.422352,0.588821,0.588821,V1 eligible vs censored/cutoff-excluded
4,avg_ride_days_per_week,STANDARDIZED_MEAN_DIFFERENCE,5.290701,4.607839,0.458694,0.458694,V1 eligible vs censored/cutoff-excluded
0,motorcycle_age_years,STANDARDIZED_MEAN_DIFFERENCE,3.621993,4.587993,-0.448563,0.448563,V1 eligible vs censored/cutoff-excluded
3,service_sequence,STANDARDIZED_MEAN_DIFFERENCE,6.287639,5.147805,0.182766,0.182766,V1 eligible vs censored/cutoff-excluded
6,adv_previous_service_count,STANDARDIZED_MEAN_DIFFERENCE,5.287639,4.147805,0.182766,0.182766,V1 eligible vs censored/cutoff-excluded
1,snapshot_odometer_km,STANDARDIZED_MEAN_DIFFERENCE,67969.048983,63003.720786,0.094165,0.094165,V1 eligible vs censored/cutoff-excluded
7,usage_type,TOTAL_VARIATION_DISTANCE,NaN,NaN,0.034340,0.034340,distribution distance


## 9. Feature importance, error analizi ve R² sınırı

### Ne yapıyoruz?
Final model için Validation'da sınırlı permutation importance hesaplıyor; en büyük 20 TEST hatasını ve servis-geçmiş derinliği hata örüntülerini çıkarıyoruz.

### Neden yapıyoruz?
Importance hangi alanın tahmine katkı sağladığını gösterir; **importance causality değildir**. Kalan hataların ilk servis, yüksek kullanım, yüksek kilometre veya oynak geçmiş aralıklarında yoğunlaşıp yoğunlaşmadığını görürüz.

### R² neden sınırlı kalabilir?
Servise geliş; kullanımın yanında müşteri tercihi, randevu, rastlantısal arıza, gecikme ve ölçülmeyen dış etkenlerden etkilenir. Observed-only seçim yanlılığı, sentetik generator randomness'i, interval variance ve açıklayıcı feature sınırı aynı X için farklı y üretebilir.


In [ ]:
importance_rows=[]
for target in ["DAYS","KM"]:
    rec=candidate_store[target][selected_name[target]]; set_name=rec["feature_set"]
    Xval=feature_set_matrix(set_name,positions[target]["VALIDATION"]); yval=ysets[target]["VALIDATION"]
    take=np.arange(min(1000,len(yval)))
    pi=permutation_importance(final_models[target],Xval[take],yval[take],scoring="neg_mean_absolute_error",n_repeats=2,random_state=SEED,n_jobs=-1)
    names=feature_set_names(set_name); order=np.argsort(pi.importances_mean)[::-1][:20]
    for rank,i in enumerate(order,1):
        importance_rows.append({"target":target,"rank":rank,"feature":names[i],"importance":float(pi.importances_mean[i]),"method":"VALIDATION_PERMUTATION_MAE"})
importance=pd.DataFrame(importance_rows)
importance.to_csv(TABLES/"v1_advanced_feature_importance.csv",index=False,encoding="utf-8-sig")

error_context_cols=["snapshot_id","motorcycle_id","usage_type","riding_intensity","brand","category","model_id","workshop_id",
                    "motorcycle_age_years","snapshot_odometer_km","service_sequence","previous_failure_count","annual_km_baseline",
                    "maintenance_overdue_days_pre_service","maintenance_overdue_km_pre_service"]
error_base=test_predictions.merge(test_context[error_context_cols],on=["snapshot_id","motorcycle_id"],validate="one_to_one")
error_base["interval_days_cv"]=eng.loc[error_base.snapshot_id,"adv_previous_interval_days_cv"].to_numpy()
error_base["interval_km_cv"]=eng.loc[error_base.snapshot_id,"adv_previous_interval_km_cv"].to_numpy()
error_base["recent_90d_km"]=eng.loc[error_base.snapshot_id,"adv_recent_90d_km"].to_numpy()
largest_days=error_base.nlargest(20,"days_abs_error").copy(); largest_days["target"]="DAYS"
largest_km=error_base.nlargest(20,"km_abs_error").copy(); largest_km["target"]="KM"
pd.concat([largest_days,largest_km],ignore_index=True).to_csv(TABLES/"v1_advanced_error_analysis.csv",index=False,encoding="utf-8-sig")

test_context["history_depth"]=pd.cut(test_context.service_sequence,[0,1,2,4,8,np.inf],labels=["FIRST","SECOND","3-4","5-8","9+"])
history_error_rows=[]
for target,col in [("DAYS","days_abs_error"),("KM","km_abs_error")]:
    temp=pd.DataFrame({"segment":test_context.history_depth,"abs_error":test_predictions[col]})
    for segment,g in temp.groupby("segment",observed=True):
        history_error_rows.append({"target":target,"history_depth":str(segment),"n":len(g),"mae":g.abs_error.mean(),"median_ae":g.abs_error.median(),"notes":"INTERPRET" if len(g)>=MIN_SEGMENT_N else "N<50"})
history_errors=pd.DataFrame(history_error_rows)
history_errors.to_csv(TABLES/"v1_advanced_history_depth_errors.csv",index=False,encoding="utf-8-sig")

# TRAIN/VAL/TEST gap final modelde ayrıca ölçülür; seçim değiştirmez.
gap_rows=[]
for target in ["DAYS","KM"]:
    set_name=candidate_store[target][selected_name[target]]["feature_set"]; X=feature_set_matrix(set_name); p=positions[target]
    train_pred=np.clip(final_models[target].predict(X[p["TRAIN"]]),0,None)
    train_mae=regression_metrics(ysets[target]["TRAIN"],train_pred,target)["mae"]
    val_mae=adv_metric(target,"mae","VALIDATION"); test_mae=adv_metric(target,"mae","TEST")
    gap_rows.append({"target":target,"train_mae":train_mae,"validation_mae":val_mae,"test_mae":test_mae,
                     "validation_minus_train":val_mae-train_mae,"test_minus_validation":test_mae-val_mae,
                     "test_gap_pct":(test_mae-val_mae)/val_mae*100,
                     "overfitting_warning":"YES" if val_mae>train_mae*1.5 and val_mae-train_mae>(5 if target=="DAYS" else 500) else "NO"})
gap_table=pd.DataFrame(gap_rows)
gap_table.to_csv(TABLES/"v1_advanced_train_validation_test_gap.csv",index=False,encoding="utf-8-sig")

top_error_pattern=(history_errors[history_errors.n>=MIN_SEGMENT_N].sort_values("mae",ascending=False).iloc[0])
display(importance.groupby("target").head(10)); display(history_errors); display(gap_table)


,target,rank,feature,importance,method
0,DAYS,1,numeric__annual_km_baseline,5.009201,VALIDATION_PERMUTATION_MAE
1,DAYS,2,numeric_structural__engine_displacement_cc,1.410400,VALIDATION_PERMUTATION_MAE
2,DAYS,3,numeric__current_inspection_finding_task_count,1.184664,VALIDATION_PERMUTATION_MAE
3,DAYS,4,numeric__highway_ratio,0.725970,VALIDATION_PERMUTATION_MAE
4,DAYS,5,categorical__customer_type_INDIVIDUAL,0.477501,VALIDATION_PERMUTATION_MAE
5,DAYS,6,categorical__riding_intensity_HIGH,0.288938,VALIDATION_PERMUTATION_MAE
6,DAYS,7,adv_rolling_min_interval_days,0.228051,VALIDATION_PERMUTATION_MAE
7,DAYS,8,adv_rolling_median_interval_days,0.212650,VALIDATION_PERMUTATION_MAE
8,DAYS,9,numeric__offroad_ratio,0.183311,VALIDATION_PERMUTATION_MAE
9,DAYS,10,categorical__current_service_type_code_PERIODIC,0.140655,VALIDATION_PERMUTATION_MAE


,target,history_depth,n,mae,median_ae,notes
0,DAYS,FIRST,163,28.158241,20.851477,INTERPRET
1,DAYS,SECOND,172,25.410199,18.688072,INTERPRET
2,DAYS,3-4,314,27.262680,18.579370,INTERPRET
3,DAYS,5-8,616,23.668094,17.342661,INTERPRET
4,DAYS,9+,1357,19.250636,14.914752,INTERPRET
5,KM,FIRST,163,1283.397803,749.360310,INTERPRET
6,KM,SECOND,172,1233.830360,782.902386,INTERPRET
7,KM,3-4,314,1312.411880,885.280134,INTERPRET
8,KM,5-8,616,1384.756469,924.285151,INTERPRET
9,KM,9+,1357,1675.136673,1280.855743,INTERPRET


,target,train_mae,validation_mae,test_mae,validation_minus_train,test_minus_validation,test_gap_pct,overfitting_warning
0,DAYS,68.581729,21.696521,22.205754,-46.885209,0.509233,2.347074,NO
1,KM,3168.305653,1473.692781,1510.175608,-1694.612872,36.482828,2.475606,NO


## 10. Grafikler, model kartı ve bilimsel karar

### Ne yapıyoruz?
Zorunlu 15 görseli, model/preprocessor artifactlarını, audit tablolarını ve ayrıntılı Markdown raporu kaydediyoruz.

### Neden yapıyoruz?
Sonuçlar yalnız notebook ekranında kalmaz; seçim mantığı, leakage kontrolleri, kalan hata ve regression ceiling kararı yeniden denetlenebilir olur.

### Leakage riski var mı?
Artifact metadata'sı TEST'in selection/tuning amacıyla kullanılmadığını ve production validation'ın BLOCKED kaldığını açıkça taşır.

### Sonuç nasıl yorumlanmalı?
R² araştırma eşikleri başarı gate'i değildir. Nihai hüküm Basic V1'e göre MAE/R² kazancı, Validation→TEST stability, ablation ve model karmaşıklığı birlikte değerlendirilerek verilir.


In [ ]:
days_improvement=(basic_metric("DAYS","mae")-adv_metric("DAYS","mae"))/basic_metric("DAYS","mae")*100
km_improvement=(basic_metric("KM","mae")-adv_metric("KM","mae"))/basic_metric("KM","mae")*100
days_r2_gain=adv_metric("DAYS","r2")-basic_metric("DAYS","r2")
km_r2_gain=adv_metric("KM","r2")-basic_metric("KM","r2")
avg_improvement=(days_improvement+km_improvement)/2
if avg_improvement>=15: final_verdict="STRONG IMPROVEMENT"
elif avg_improvement>=7: final_verdict="MODERATE IMPROVEMENT"
elif avg_improvement>1: final_verdict="SMALL IMPROVEMENT"
else: final_verdict="NO IMPROVEMENT"
if avg_improvement>=10 and max(days_r2_gain,km_r2_gain)>=.08: regression_ceiling="SIGNIFICANT HEADROOM REMAINS"
elif avg_improvement>=2: regression_ceiling="DIMINISHING RETURNS"
else: regression_ceiling="PRACTICAL CEILING REACHED"

def selected_description(target):
    rec=candidate_store[target][selected_name[target]]
    if rec["family"]=="ENSEMBLE": return {"type":"ENSEMBLE","members":rec["members"],"weights":rec["weights"]}
    return {"type":"SINGLE","family":rec["family"],"params":rec["params"],"target_transform":rec["transform"]}

model_card={
    "dataset_version":DATASET_VERSION,"feature_set_version":"V1_ADVANCED_FEATURES_1.0.0",
    "preprocessing_version":"V1_ADVANCED_PREPROCESSOR_1.0.0","python_version":platform.python_version(),"sklearn_version":sklearn_version,
    "days_model":selected_description("DAYS"),"km_model":selected_description("KM"),
    "days_hyperparameters":selected_description("DAYS"),"km_hyperparameters":selected_description("KM"),
    "target_transform":{"days":candidate_store["DAYS"][selected_name["DAYS"]]["transform"],"km":candidate_store["KM"][selected_name["KM"]]["transform"]},
    "validation_selection":"Feature set, family, transform and blend selected by full Validation actionable MAE; TEST used once after lock",
    "train_rows":{"days":len(ysets['DAYS']['TRAIN']),"km":len(ysets['KM']['TRAIN'])},"test_rows":{"days":len(ysets['DAYS']['TEST']),"km":len(ysets['KM']['TEST'])},
    "feature_set":{"days":candidate_store['DAYS'][selected_name['DAYS']]['feature_set'],"km":candidate_store['KM'][selected_name['KM']]['feature_set']},
    "final_feature_count":{"days":len(feature_set_names(candidate_store['DAYS'][selected_name['DAYS']]['feature_set'])),"km":len(feature_set_names(candidate_store['KM'][selected_name['KM']]['feature_set']))},
    "random_seed":SEED,"reproducibility":reproducibility,"regression_ceiling":regression_ceiling,
    "limitations":["Synthetic v1.2 offline PoC","Observed-only selection bias","Censored targets excluded","No production validation","Importance is not causality"],
    "production_validation_status":"BLOCKED",
}
with open(MODELS/"v1_advanced_regression_model_card.json","w",encoding="utf-8") as f: json.dump(model_card,f,ensure_ascii=False,indent=2,default=str)

plt.style.use("seaborn-v0_8-whitegrid")
def savefig(name):
    plt.tight_layout(); plt.savefig(FIGURES/name,dpi=150,bbox_inches="tight"); plt.close()

for target,col,file in [("DAYS","days_val_mae","01_feature_set_ablation_days.png"),("KM","km_val_mae","02_feature_set_ablation_km.png")]:
    plt.figure(figsize=(9,4)); plt.plot(ablation.feature_set,ablation[col],marker="o"); plt.xticks(rotation=25,ha="right"); plt.ylabel("Validation MAE"); plt.title(f"{target} Feature Set Ablation"); savefig(file)
for target,file in [("DAYS","03_days_model_validation_comparison.png"),("KM","04_km_model_validation_comparison.png")]:
    p=candidate_rankings[target].head(12).sort_values("mae"); plt.figure(figsize=(10,6)); plt.barh(p.candidate,p.mae); plt.xlabel("Validation MAE"); plt.title(f"{target} Model / Transform / Blend"); savefig(file)
for target,file in [("DAYS","05_v0_v1_advanced_days.png"),("KM","06_v0_v1_advanced_km.png")]:
    p=comparison[(comparison.target==target)&(comparison.metric=="mae")]; plt.figure(figsize=(7,4)); plt.bar(p.model,p.value,color=["#888","#4c78a8","#f58518"]); plt.ylabel("TEST MAE"); plt.title(f"V0 vs V1 Basic vs Advanced — {target}"); savefig(file)
for target,y,pred,file in [("DAYS",ysets["DAYS"]["TEST"],final_predictions[("DAYS","TEST")],"07_actual_vs_predicted_days.png"),("KM",ysets["KM"]["TEST"],final_predictions[("KM","TEST")],"08_actual_vs_predicted_km.png")]:
    plt.figure(figsize=(6,5)); plt.scatter(y,pred,s=8,alpha=.3); lim=max(np.max(y),np.max(pred)); plt.plot([0,lim],[0,lim],"r--"); plt.xlabel("Actual"); plt.ylabel("Predicted"); plt.title(f"Actual vs Predicted {target}"); savefig(file)
for target,y,pred,file in [("DAYS",ysets["DAYS"]["TEST"],final_predictions[("DAYS","TEST")],"09_days_residuals.png"),("KM",ysets["KM"]["TEST"],final_predictions[("KM","TEST")],"10_km_residuals.png")]:
    residual=pred-y; plt.figure(figsize=(7,4)); plt.scatter(pred,residual,s=8,alpha=.3); plt.axhline(0,color="r",ls="--"); plt.xlabel("Prediction"); plt.ylabel("Residual (pred-actual)"); plt.title(f"{target} Residuals"); savefig(file)
for target,file in [("DAYS","11_days_error_by_history_depth.png"),("KM","12_km_error_by_history_depth.png")]:
    p=history_errors[history_errors.target==target]; plt.figure(figsize=(7,4)); plt.bar(p.history_depth,p.mae); plt.ylabel("TEST MAE"); plt.title(f"{target} Error by History Depth"); savefig(file)
for target,file in [("DAYS","13_days_feature_importance.png"),("KM","14_km_feature_importance.png")]:
    p=importance[importance.target==target].head(15).sort_values("importance"); plt.figure(figsize=(10,6)); plt.barh(p.feature,p.importance); plt.xlabel("Validation MAE increase when shuffled"); plt.title(f"{target} Feature Importance"); savefig(file)
p=population_shift.head(10).sort_values("abs_shift"); plt.figure(figsize=(9,5)); plt.barh(p.feature,p.abs_shift); plt.xlabel("Absolute SMD / TVD"); plt.title("Observed vs Censored/Cutoff Population Shift"); savefig("15_observed_vs_censored_feature_shift.png")

limited_diagnostic=[]
if adv_metric("DAYS","r2")<.60: limited_diagnostic.append("DAYS R²<0.60: interval variance, müşteri gecikmesi, arıza rastlantısallığı ve observed-only selection incelenmelidir.")
if adv_metric("KM","r2")<.45: limited_diagnostic.append("KM R²<0.45: kullanım trendi dışındaki ölçülmeyen rota/aktivite ve failure etkileri sınır oluşturabilir.")
why_r2=" ".join(limited_diagnostic) if limited_diagnostic else "Araştırma eşikleri aşıldı; yine de production genellemesi iddia edilmez."

report=f"""# Executive Summary

Advanced V1 tamamlandı. DAYS final: **{selected_name['DAYS']}**; KM final: **{selected_name['KM']}**. Basic V1'e göre TEST MAE değişimi DAYS {days_improvement:.2f}%, KM {km_improvement:.2f}%. Production validation **BLOCKED**.

# Why Advanced Regression

Amaç leakage-free snapshot/history featurelarıyla observed-only regression sınırını ölçmektir; R² hedefi zorlanmamıştır.

# Existing V1 Baseline

DAYS Basic MAE/R²: {basic_metric('DAYS','mae'):.3f}/{basic_metric('DAYS','r2'):.3f}. KM: {basic_metric('KM','mae'):.3f}/{basic_metric('KM','r2'):.3f}.

# Feature Engineering

Var olan 05 featureları tekrar üretilmedi. {eng.shape[1]} yeni ham feature; service rolling 2/5/expanding, tamamlanmış-ay usage trendleri, maintenance/failure oranları, policy-derived due ve sınırlı interaction gruplarında üretildi.

# Leakage Controls

Her engineered feature `period/service <= snapshot_at` sözleşmesindedir. Policy due, actual next service değildir. Audit: {len(feature_audit)}/{len(feature_audit)} PASS. TEST tuning/seçimde kullanılmadı.

# Feature Ablation

{ablation.to_markdown(index=False)}

# Target Distribution

{target_distribution.to_markdown(index=False)}

# Target Transformation

DAYS: {candidate_store['DAYS'][selected_name['DAYS']]['transform']}; KM: {candidate_store['KM'][selected_name['KM']]['transform']}. Raw ve log1p aynı Validation metrikleriyle karşılaştırıldı.

# Model Families

HistGradientBoosting, ExtraTrees, RandomForest, GradientBoosting, XGBoost, LightGBM ve CatBoost değerlendirildi. Availability/training durumu `v1_advanced_model_status.csv` içindedir.

# Hyperparameter Tuning

TRAIN zamana göre sıralandı; iki expanding fold ve her ailede iki kontrollü RandomizedSearchCV örneği kullanıldı. Primary CV metric negative MAE'dir.

# Ensemble Experiments

{ensemble_results.to_markdown(index=False) if len(ensemble_results) else 'Ensemble için yeterli başarılı farklı aile oluşmadı.'}

# Validation Selection

DAYS Validation MAE: {adv_metric('DAYS','mae','VALIDATION'):.3f}; KM: {adv_metric('KM','mae','VALIDATION'):.3f}. Feature set/model/transform/blend seçimleri TEST açılmadan kilitlendi.

# Final Test Results

DAYS MAE/Median AE/RMSE/R²: {adv_metric('DAYS','mae'):.3f}/{adv_metric('DAYS','median_ae'):.3f}/{adv_metric('DAYS','rmse'):.3f}/{adv_metric('DAYS','r2'):.3f}. KM: {adv_metric('KM','mae'):.3f}/{adv_metric('KM','median_ae'):.3f}/{adv_metric('KM','rmse'):.3f}/{adv_metric('KM','r2'):.3f}.

# V0 vs V1 vs Advanced V1

{comparison.pivot_table(index=['target','metric'],columns='model',values='value').reset_index().to_markdown(index=False)}

# Feature Importance

Top 20 permutation importance kaydedildi. Importance, causality değildir.

# Error Analysis

En büyük 20 DAYS/KM hatası ve history-depth segmentleri kaydedildi. En yüksek yeterli örnekli pattern: {top_error_pattern['target']} / {top_error_pattern['history_depth']} (n={int(top_error_pattern['n'])}, MAE={top_error_pattern['mae']:.2f}).

# Observed vs Censored Population Shift

Shift durumu: **{shift_status}**. En büyük fark: {population_shift.iloc[0]['feature']} ({population_shift.iloc[0]['metric']}={population_shift.iloc[0]['shift']:.3f}). Bu nedenle observed-only performans tüm population'a doğrudan genellenemez.

# Why R² Is Limited

{why_r2} Target noise, stochastic service behavior, missing explanatory variables, unpredictable failures, generator randomness, interval variance ve observed-only seçim etkisi birlikte değerlendirilmelidir.

# Regression Ceiling Assessment

**{regression_ceiling}**. Ortalama MAE iyileşmesi {avg_improvement:.2f}%, R² kazancı DAYS {days_r2_gain:.3f}, KM {km_r2_gain:.3f}; Validation→TEST farkları `v1_advanced_train_validation_test_gap.csv` içindedir.

# V2 Survival Motivation

V1 yalnız observed satırları regression targetı yapar. V2, 41.518 snapshotın observed+censored duration bilgisini birlikte kullanabildiği ve population shift {shift_status.lower()} olduğu için hâlâ önerilir.

# Limitations

Sentetik v1.2 offline PoC; production validation yoktur. Validation tekrar model/feature/blend seçiminde kullanılmıştır. Censored targetlar V1 metriğine dahil değildir. Feature importance nedensellik değildir.

# Final Verdict

**{final_verdict}**. R² skorunu yükseltmek için future/target feature, TEST tuning, target uydurma veya outlier silme yapılmadı.
"""
(REPORTS/"v1_advanced_regression_report.md").write_text(report,encoding="utf-8")

qa=pd.DataFrame([
    {"check":"dataset_guard","status":"PASS","evidence":DATASET_VERSION},
    {"check":"observed_masks","status":"PASS","evidence":str(days_counts)},
    {"check":"feature_leakage_audit","status":"PASS","evidence":f"{len(feature_audit)}/{len(feature_audit)}"},
    {"check":"train_only_advanced_preprocessing","status":"PASS","evidence":"fit on TRAIN features only"},
    {"check":"temporal_cv","status":"PASS","evidence":"2 expanding TRAIN folds"},
    {"check":"test_excluded_from_selection","status":"PASS","evidence":"selection locked before TEST cell"},
    {"check":"reproducibility_days","status":"PASS" if reproducibility['DAYS'] else "FAIL","evidence":"VALIDATION allclose 1e-7"},
    {"check":"reproducibility_km","status":"PASS" if reproducibility['KM'] else "FAIL","evidence":"VALIDATION allclose 1e-7"},
    {"check":"source_data_immutable","status":"PASS","evidence":"source paths opened read-only"},
    {"check":"production_validation","status":"BLOCKED","evidence":"No production extract"},
])
qa.to_csv(TABLES/"v1_advanced_qa.csv",index=False,encoding="utf-8-sig")
if (qa[qa.check!="production_validation"].status!="PASS").any(): raise RuntimeError("Advanced QA failed")

required=[MODELS/"v1_advanced_preprocessor.joblib",MODELS/"v1_advanced_days_model.joblib",MODELS/"v1_advanced_km_model.joblib",
          MODELS/"v1_advanced_regression_model_card.json",OUTPUTS/"v1_advanced_regression_test_predictions.parquet",
          TABLES/"v1_advanced_feature_ablation.csv",TABLES/"v1_advanced_model_tuning_days.csv",TABLES/"v1_advanced_model_tuning_km.csv",
          TABLES/"v1_advanced_ensemble_results.csv",TABLES/"v1_advanced_feature_leakage_audit.csv",REPORTS/"v1_advanced_regression_report.md"]
required += [FIGURES/f"{i:02d}_{name}" for i,name in enumerate([
    "feature_set_ablation_days.png","feature_set_ablation_km.png","days_model_validation_comparison.png","km_model_validation_comparison.png",
    "v0_v1_advanced_days.png","v0_v1_advanced_km.png","actual_vs_predicted_days.png","actual_vs_predicted_km.png","days_residuals.png","km_residuals.png",
    "days_error_by_history_depth.png","km_error_by_history_depth.png","days_feature_importance.png","km_feature_importance.png","observed_vs_censored_feature_shift.png"],1)]
missing=[str(p) for p in required if not p.exists()]
if missing: raise RuntimeError(f"Missing advanced artifacts: {missing}")
print("V1_ADVANCED_REGRESSION_STATUS=PASS")
print("SELECTED=",selected_name,"CEILING=",regression_ceiling,"VERDICT=",final_verdict)


V1_ADVANCED_REGRESSION_STATUS=PASS
SELECTED= {'DAYS': 'ENSEMBLE__INVERSE_VALIDATION_MAE_BLEND', 'KM': 'GradientBoostingRegressor__RAW'} CEILING= DIMINISHING RETURNS VERDICT= SMALL IMPROVEMENT


## Basit sonuç

Bu çalışma, yalnız snapshot anında bilinen bilgilerle daha ayrıntılı geçmiş ve kullanım özetleri oluşturdu. Modeller geçmiş TRAIN dönemlerinde ayarlandı, sonraki VALIDATION döneminde seçildi ve TEST yalnız final kararından sonra açıldı. Daha yüksek R² uğruna gelecek servis bilgisi, uydurma censored hedef, TEST ayarı veya outlier silme kullanılmadı.
